# BMIN 5200 — Week 7 in-class exercise
## CLIPS and clipspy: the transfusion rules

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week07.ipynb)

**Time:** ~25 minutes · **Pairs with:** Building an expert system (CLIPS, clipspy)

### Tasks
- Build a blood transfusion eligibility expert system in CLIPS from an empty environment: template, deffacts, rules
- Watch the fact base and the agenda before and after `env.run()`, so the recognize-act cycle is visible rather than described
- Add two rules of your own to the rule base and check them against a ward census
- Work through the three things that confused this class last year, which are all about how CLIPS and Python fit together rather than about rules

### Background
Last week you wrote a chaining engine by hand, so nothing in CLIPS should be conceptually new: the agenda is your conflict set, `assert` adds to your fact base, and `run` is your recognize-act loop. What CLIPS adds is pattern matching with variable binding, an efficient match algorithm, and thirty years of hardening. Transfusion thresholds are a genuinely good fit for a rule system — they are written as thresholds, they are audited, and the answer has to be explainable to a blood bank.

## Setup

`clipspy` is not preinstalled in Colab. It ships Linux wheels, so the install takes a few seconds
and needs no compiler. Everything after this cell is real CLIPS running inside the notebook.

In [ ]:
%pip install -q clipspy
import clips

# If this cell fails in Colab, run it once more; the wheel occasionally needs a second attempt.
print(f"clipspy {clips.__version__} imported")
print(clips.Environment().eval('(str-cat "the CLIPS engine is reachable from Python")'))

## Part 1 — Environments, templates, and facts

`clips.Environment()` is the whole expert system: rule base, fact base, and inference engine in
one object, matching the "The clipspy environment" slide. Everything below is built by handing
CLIPS source text to `env.build()`. The `deftemplate` declares the shape of a patient fact, which
is what lets a rule pattern say "the hemoglobin slot" instead of counting positions, and the
`deffacts` names a set of facts that get asserted every time you call `env.reset()`.

In [ ]:
env = clips.Environment()

env.build("""
(deftemplate patient
  (slot name (type STRING))
  (slot hemoglobin (type FLOAT))
  (slot active_bleeding (type SYMBOL) (allowed-values yes no) (default no))
  (slot acute_coronary_syndrome (type SYMBOL) (allowed-values yes no) (default no))
  (slot symptomatic (type SYMBOL) (allowed-values yes no) (default no))
  (slot consent (type SYMBOL) (allowed-values given refused unable) (default given)))
""")

env.build("""
(deffacts ward-census
  (patient (name "Okafor")  (hemoglobin 6.4))
  (patient (name "Delgado") (hemoglobin 7.8) (active_bleeding yes))
  (patient (name "Ruiz")    (hemoglobin 6.9) (consent refused))
  (patient (name "Bhatt")   (hemoglobin 7.6) (acute_coronary_syndrome yes))
  (patient (name "Novak")   (hemoglobin 7.9) (symptomatic yes))
  (patient (name "Iyer")    (hemoglobin 6.2) (consent unable)))
""")


def show_facts(env, title):
    facts = list(env.facts())
    print(f"{title}  ({len(facts)} facts)")
    for fact in facts:
        print(f"  {fact}")
    print()


show_facts(env, "immediately after build, before reset")
env.reset()                       # reset asserts everything in every deffacts
show_facts(env, "after env.reset()")

Two things to notice. Before `reset()` the fact base is empty: a `deffacts` is a *declaration*
of facts, not an assertion of them. And after `reset()` every patient shows all six slots even
though the census only supplied two or three, because the template's `(default ...)` filled the
rest in. That default is doing real clinical work — it is asserting that a patient with no
recorded bleeding is not bleeding, which is an assumption you would want written down somewhere
in a real system.

Now a rule. The pattern `(hemoglobin ?hgb&:(< ?hgb 7.0))` binds the slot to the variable `?hgb`
and then constrains it, which is the variable binding from last week done properly. Before running,
look at `env.activations()`: that is the agenda, the set of rule-plus-fact combinations that are
ready to fire, and it is exactly the conflict set you implemented by hand in Week 6.

In [ ]:
env.build("""
(defrule transfuse-severe-anemia
  (patient (name ?name) (hemoglobin ?hgb&:(< ?hgb 7.0)) (consent given))
  =>
  (assert (recommendation ?name transfuse "hemoglobin below the 7 g/dL restrictive threshold")))
""")

print("agenda before run:")
for activation in env.activations():
    print(f"  {activation}")

rules_fired = env.run()

print(f"\nenv.run() returned: {rules_fired}")
print("agenda after run:", list(env.activations()))
print()
show_facts(env, "fact base after run")

One rule matched one patient, so `run()` returned 1 and a `recommendation` fact appeared.
Ruiz has a hemoglobin of 6.9, below the same threshold, and did not get a recommendation, because
the rule also requires `(consent given)` and Ruiz declined. Iyer at 6.2 is in the same position
with `consent unable`. Neither of those is a bug; both are cases your rule base has not covered
yet, which is what Part 2 is for.

## Part 2 — Completing the transfusion rule base

Here are three more rules covering the situations where a restrictive threshold of 7 g/dL does
not apply — active bleeding and acute coronary syndrome both justify transfusing higher — plus a
rule that records a hold when consent was refused, and one that recommends observation for a
stable patient. Read them before running; the pattern syntax is the same in all of them.

In [ ]:
TRANSFUSE_ACTIVE_BLEEDING = """
(defrule transfuse-active-bleeding
  (patient (name ?name) (hemoglobin ?hgb&:(< ?hgb 9.0)) (active_bleeding yes) (consent given))
  =>
  (assert (recommendation ?name transfuse "active bleeding with hemoglobin below 9 g/dL")))
"""

TRANSFUSE_ACS = """
(defrule transfuse-acute-coronary-syndrome
  (patient (name ?name) (hemoglobin ?hgb&:(< ?hgb 8.0)) (acute_coronary_syndrome yes) (consent given))
  =>
  (assert (recommendation ?name transfuse "acute coronary syndrome raises the threshold to 8 g/dL")))
"""

HOLD_CONSENT_REFUSED = """
(defrule hold-consent-refused
  (patient (name ?name) (consent refused))
  =>
  (assert (recommendation ?name hold "patient declined transfusion")))
"""

OBSERVE_STABLE = """
(defrule observe-stable
  (patient (name ?name) (hemoglobin ?hgb&:(>= ?hgb 7.0))
           (active_bleeding no) (acute_coronary_syndrome no) (symptomatic no) (consent given))
  =>
  (assert (recommendation ?name observe "above the restrictive threshold and not bleeding")))
"""

for rule_source in (TRANSFUSE_ACTIVE_BLEEDING, TRANSFUSE_ACS, HOLD_CONSENT_REFUSED, OBSERVE_STABLE):
    env.build(rule_source)

print("rules now in the rule base:")
for rule in env.rules():
    print(f"  {rule.name}")

Two patients are still uncovered. Novak is at 7.9 g/dL and symptomatic — above the restrictive
threshold, but symptomatic anemia is an accepted indication to transfuse. Iyer is at 6.2 g/dL with
`consent unable`, which is not a refusal; it means nobody has been able to obtain consent and a
surrogate decision maker has to be found. Write those two rules.

Both skeletons below build and run as they stand, but neither can ever fire: each one opens with
a pattern `(todo-not-finished)`, a fact that is never asserted. Delete that line and complete the
patient pattern.

### Predict before you run

Fill in both rules, then commit to answers for these before running the cell:

1. Okafor is at 6.4 g/dL with consent given, and exactly one rule matches. How many
   `recommendation` facts should Okafor end up with?
2. Delgado is at 7.8 g/dL and actively bleeding, so `transfuse-active-bleeding` matches. Does
   `observe-stable` match Delgado as well? Why not?
3. With both of your rules working, how many rules should `env.run()` report having fired across
   the whole six-patient census?

In [ ]:
STUDENT_RULE_SYMPTOMATIC = """
(defrule transfuse-symptomatic-anemia
  (todo-not-finished)
  ;; TODO: match a patient whose hemoglobin is below 8.0, whose symptomatic slot is yes, and
  ;;       whose consent is given. Delete the (todo-not-finished) line above.
  (patient (name ?name))
  =>
  (assert (recommendation ?name transfuse "symptomatic anemia below 8 g/dL")))
"""

STUDENT_RULE_SURROGATE = """
(defrule consent-unable-find-surrogate
  (todo-not-finished)
  ;; TODO: match a patient whose consent slot is unable. Delete the (todo-not-finished) line.
  (patient (name ?name))
  =>
  (assert (recommendation ?name defer "consent not obtainable; contact surrogate decision maker")))
"""

env.build(STUDENT_RULE_SYMPTOMATIC)
env.build(STUDENT_RULE_SURROGATE)

env.reset()                       # start the census over, with the full rule base in place
rules_fired = env.run()

print(f"rules fired: {rules_fired}")
print()
print(f"  {'patient':<10}  {'action':<10}  reason")
recommendations = {}
for fact in env.facts():
    if fact.template.name == "recommendation":
        recommendations[str(fact[0])] = (str(fact[1]), str(fact[2]))

for name in ("Okafor", "Delgado", "Ruiz", "Bhatt", "Novak", "Iyer"):
    if name in recommendations:
        action, reason = recommendations[name]
        print(f"  {name:<10}  {action:<10}  {reason}")
    else:
        print(f"  {name:<10}  {'-':<10}  no rule covers this patient yet")

In [ ]:
EXPECTED = {
    "Okafor":  "transfuse",   # 6.4 g/dL, below the restrictive threshold
    "Delgado": "transfuse",   # 7.8 g/dL but actively bleeding
    "Ruiz":    "hold",        # 6.9 g/dL but declined
    "Bhatt":   "transfuse",   # 7.6 g/dL with acute coronary syndrome
    "Novak":   "transfuse",   # 7.9 g/dL but symptomatic  -> your first rule
    "Iyer":    "defer",       # 6.2 g/dL, consent unable   -> your second rule
}

print(f"  {'patient':<10}  {'expected':<10}  {'produced':<10}  ")
for name, expected_action in EXPECTED.items():
    produced = recommendations.get(name, ("(none)", ""))[0]
    mark = "ok" if produced == expected_action else "<-- not yet"
    print(f"  {name:<10}  {expected_action:<10}  {produced:<10}  {mark}")

## Part 3 — Common errors

These three cost this class several hours last year, and none of them is about rules. They are all
about the seam between CLIPS and Python. Read this section carefully even if your rules already
work.

### Error 1 — `env.run()` returns a count, not your output

`env.run()` returns an integer: the number of rule activations that fired. It has nothing to do
with what your rules printed. If a rule does `(printout t "TRANSFUSE " ?name crlf)` and you see
`1` in your notebook, the rule worked; you are looking at the return value, not the printout.

The printout itself goes to the CLIPS logical name `stdout`, which clipspy routes to the
process's real standard output. In a notebook that is the kernel's stdout, not the cell's output
area, so depending on how you are running — Colab, Jupyter, a terminal — it may appear in a server
log, appear out of order, or not appear at all. Run this and watch where the text lands.

In [ ]:
printing_env = clips.Environment()
printing_env.build('(defrule announce (screening-complete) => (printout t "SCREENING COMPLETE" crlf))')
printing_env.assert_string("(screening-complete)")

result = printing_env.run()
print(f"env.run() returned {result!r}, of type {type(result).__name__}")
print("That 1 means one rule fired. It is not the text the rule printed.")

There are two reliable fixes and you should prefer the first.

**Assert facts instead of printing them.** A rule that asserts `(recommendation ?name transfuse
"...")` puts its conclusion in the fact base where Python can read it, query it, and write it to a
table. That is what every rule in Part 2 does, and it is why the census table above works. A rule
that only prints has thrown its conclusion away.

**Or attach a router to capture the output.** A router is CLIPS's output redirection mechanism.
Subclass `clips.Router`, say which logical name you want to intercept, and collect what gets
written. Use this when you are debugging, or when you inherited rules full of `printout` that you
do not want to rewrite.

In [ ]:
import io


class TranscriptRouter(clips.Router):
    """Intercepts everything CLIPS writes to 'stdout' and keeps it in a Python string buffer."""

    def __init__(self):
        super().__init__("transcript", 40)     # 40 is a priority; higher intercepts first
        self.transcript = io.StringIO()

    def query(self, logical_name):
        return logical_name == "stdout"        # yes, this router handles stdout

    def write(self, logical_name, message):
        self.transcript.write(message)


captured_env = clips.Environment()
transcript_router = TranscriptRouter()
captured_env.add_router(transcript_router)

captured_env.build("""
(defrule announce-two-lines
  (screening-complete)
  =>
  (printout t "unit 1 crossmatched" crlf)
  (printout t "unit 2 crossmatched" crlf))
""")
captured_env.assert_string("(screening-complete)")

rules_fired = captured_env.run()
print(f"env.run() returned: {rules_fired}")
print("what the rule actually printed, now available in Python:")
print(transcript_router.transcript.getvalue())

### Error 2 — `(start_exclusion)`: patterns with no deftemplate are control facts

In the Homework 3 starter code you will see a rule whose first pattern is something like
`(start_exclusion)`, and you will not find a `deftemplate` for it anywhere. That is not an
oversight. CLIPS has two kinds of facts: *ordered* facts, which are just a list of fields and need
no template at all, and *unordered* (template) facts like our `patient`, which do. A bare
`(start_exclusion)` is an ordered fact with a single field and zero data in it.

Its entire job is sequencing. A rule that opens with a control fact cannot fire until something
asserts that fact, which lets you say "run all the inclusion rules first, and only then start the
exclusion checks". Below, the screening rule sits on the agenda doing nothing until the control
fact arrives.

In [ ]:
sequencing_env = clips.Environment()
sequencing_env.build("(deftemplate candidate (slot name) (slot hemoglobin))")

# Phase 1: decide whether the patient meets the transfusion indication, then open the next phase.
sequencing_env.build("""
(defrule check-indication
  (candidate (name ?name) (hemoglobin ?hgb&:(< ?hgb 7.0)))
  =>
  (assert (indication-met ?name))
  (assert (start_exclusion)))
""")

# Phase 2: this rule cannot fire until phase 1 has asserted the control fact. Note that
# (start_exclusion) has no deftemplate anywhere in this program, and does not need one.
sequencing_env.build("""
(defrule check-exclusions
  (start_exclusion)
  (indication-met ?name)
  =>
  (assert (cleared-for-crossmatch ?name)))
""")

sequencing_env.assert_string('(candidate (name "Okafor") (hemoglobin 6.4))')

print(f"agenda before any run: {[str(a) for a in sequencing_env.activations()]}")
print("Only check-indication is on the agenda; check-exclusions is waiting on the control fact.")
print()

print(f"rules fired: {sequencing_env.run()}")
for fact in sequencing_env.facts():
    print(f"  {fact}")

print()
print("templates CLIPS knows about:")
print(f"  {[t.name for t in sequencing_env.templates()]}")
print("start_exclusion appears there as an implied template that CLIPS created for itself.")

Two reasons this idiom is everywhere in expert systems. First, real protocols have phases —
screen, then confirm, then check exclusions — and a rule base has no built-in notion of order, so
you encode order as data. Second, it is how you keep rules from firing on incomplete information:
nothing checks exclusion criteria until the facts the exclusion rules depend on have actually been
established. CLIPS also offers `salience` and `focus` for this, but control facts are the
portable way and the one Homework 3 uses.

### Error 3 — the defrule name and the Python variable name are two different namespaces

This is the one that generated the most confusion. When you write

```python
DEFRULE_NOT_ELIGIBLE_INCLUSION = """(defrule not_eligible_exclusion ...)"""
```

there is no mismatch to fix. `DEFRULE_NOT_ELIGIBLE_INCLUSION` is a Python variable holding a
string. CLIPS never sees it. The moment you call `env.build(...)`, CLIPS parses the *text* and
registers a rule under the name written after `defrule` — here `not_eligible_exclusion`. Python's
name and CLIPS's name have nothing to do with each other, and CLIPS would behave identically if
you had passed the string as a literal with no variable at all.

In [ ]:
namespace_env = clips.Environment()
namespace_env.build("(deftemplate candidate (slot name))")

DEFRULE_NOT_ELIGIBLE_INCLUSION = """
(defrule not_eligible_exclusion
  (candidate (name ?name))
  =>
  (assert (excluded ?name "version A")))
"""
namespace_env.build(DEFRULE_NOT_ELIGIBLE_INCLUSION)

print("rules CLIPS knows about:", [rule.name for rule in namespace_env.rules()])
print("Note what is NOT in that list: DEFRULE_NOT_ELIGIBLE_INCLUSION.")

try:
    namespace_env.find_rule("DEFRULE_NOT_ELIGIBLE_INCLUSION")
except LookupError as error:
    print(f"find_rule('DEFRULE_NOT_ELIGIBLE_INCLUSION') -> LookupError: {error}")

print(f"find_rule('not_eligible_exclusion')            -> {namespace_env.find_rule('not_eligible_exclusion').name}")

The real hazard runs the other way. Because CLIPS keys rules on the name inside the string, two
*different* Python variables holding two *different* rules with the same `defrule` name are the
same rule to CLIPS, and the second `build` silently replaces the first. No error, no warning. If
you copy a rule to make a variant and forget to change the name after `defrule`, your first rule
is gone. Predict what the fact base holds after the next cell before you run it.

In [ ]:
DEFRULE_EXCLUSION_VARIANT = """
(defrule not_eligible_exclusion
  (candidate (name ?name))
  =>
  (assert (excluded ?name "version B")))
"""
namespace_env.build(DEFRULE_EXCLUSION_VARIANT)

print("rules after building a second rule with the same defrule name:")
print(f"  {[rule.name for rule in namespace_env.rules()]}")

namespace_env.assert_string('(candidate (name "Okafor"))')
namespace_env.run()

print("\nfacts:")
for fact in namespace_env.facts():
    print(f"  {fact}")
print()
print("Only 'version B' was asserted. Version A was not overridden at run time; it stopped")
print("existing the moment the second build parsed a defrule with a name already in use.")

## Part 4 — Homework 3

Homework 3 is the same exercise with higher stakes: an HIV clinical trial enrollment system in
CLIPS, with inclusion criteria and exclusion criteria as separate rule phases. The shape is
already familiar — a `deftemplate` for a candidate, `deffacts` for the screening log, rules that
test thresholds, and a control fact separating the inclusion phase from the exclusion phase, which
is exactly the `(start_exclusion)` pattern from Gotcha 2.

Here is a starter you can build on. It has one inclusion rule and one exclusion rule, and it shows
the phase separation. The interesting design question, which the homework will press on, is what
the system should say about a candidate who meets every inclusion criterion and one exclusion
criterion at the same time.

In [ ]:
trial_env = clips.Environment()

trial_env.build("""
(deftemplate candidate
  (slot name (type STRING))
  (slot age (type INTEGER))
  (slot cd4_count (type INTEGER))
  (slot viral_load (type INTEGER))
  (slot pregnant (type SYMBOL) (allowed-values yes no unknown) (default unknown))
  (slot active_tuberculosis (type SYMBOL) (allowed-values yes no) (default no)))
""")

trial_env.build("""
(deffacts screening-log
  (candidate (name "P-014") (age 34) (cd4_count 180) (viral_load 40000) (pregnant no))
  (candidate (name "P-021") (age 29) (cd4_count 410) (viral_load 900)   (pregnant no))
  (candidate (name "P-033") (age 41) (cd4_count 160) (viral_load 62000) (active_tuberculosis yes)))
""")

trial_env.build("""
(defrule inclusion-cd4-and-viral-load
  (candidate (name ?name) (age ?age&:(>= ?age 18))
             (cd4_count ?cd4&:(< ?cd4 200))
             (viral_load ?vl&:(> ?vl 10000)))
  =>
  (assert (meets_inclusion ?name))
  (assert (start_exclusion)))
""")

trial_env.build("""
(defrule exclusion-active-tuberculosis
  (start_exclusion)
  (candidate (name ?name) (active_tuberculosis yes))
  =>
  (assert (meets_exclusion ?name "active tuberculosis")))
""")

trial_env.reset()
print(f"rules fired: {trial_env.run()}")
print()
screening_results = [str(f) for f in trial_env.facts()
                     if f.template.name in ("meets_inclusion", "meets_exclusion")]
for line in sorted(screening_results):
    print(f"  {line}")

print()
print("P-014 meets inclusion and no exclusion. P-021 meets neither. P-033 meets both,")
print("and nothing in this rule base yet decides what that means.")

## Discussion

1. Our template defaults `active_bleeding` to `no` when the census does not say. In a real system
   that default turns "not documented" into "not happening". Where else in this rule base does an
   absent fact get silently treated as a negative fact, and what would it take to distinguish
   "no" from "unknown"?

2. Rules that print their conclusions and rules that assert their conclusions look almost
   identical in the source. What can you do with the second kind that you cannot do with the
   first? Consider auditing, testing, and what happens when the blood bank asks why a unit was
   ordered on Tuesday.

3. P-033 satisfies every inclusion criterion and one exclusion criterion. A rule base will happily
   assert both facts and let them sit there. Who decides which one wins — the rule author with a
   salience value, the person reading the output, or a rule you have not written yet? What would
   Week 6's conflict resolution strategies say?

## Solutions

Completed versions of the two rules from Part 2. These are markdown, not code cells.

**Transfusion for symptomatic anemia**

```python
STUDENT_RULE_SYMPTOMATIC = """
(defrule transfuse-symptomatic-anemia
  (patient (name ?name) (hemoglobin ?hgb&:(< ?hgb 8.0)) (symptomatic yes) (consent given))
  =>
  (assert (recommendation ?name transfuse "symptomatic anemia below 8 g/dL")))
"""
```

**Consent not obtainable**

```python
STUDENT_RULE_SURROGATE = """
(defrule consent-unable-find-surrogate
  (patient (name ?name) (consent unable))
  =>
  (assert (recommendation ?name defer "consent not obtainable; contact surrogate decision maker")))
"""
```

Rebuilding a rule with the same `defrule` name replaces the old one, which is exactly what you
want here: fix the rule text, rerun the build cell, then `env.reset()` and `env.run()` again. You
do not need a fresh `Environment` — although if the fact base gets confusing, making one is
cheap.

**Optional — calling a Python function from inside a rule.** This is the "clipspy: Calling Python
methods from within a rule" slide, and it is how you would look up a real blood bank inventory:

```python
def units_available(blood_type):
    """Stand-in for a blood bank inventory query."""
    return {"O-negative": 2, "A-positive": 9}.get(str(blood_type), 0)

env.define_function(units_available)

env.build("""
(defrule check-inventory
  (recommendation ?name transfuse ?reason)
  =>
  (assert (inventory ?name (units_available "O-negative"))))
""")
```

The CLIPS-visible name is the Python function's `__name__`, which is a third namespace to keep
straight alongside the two in Gotcha 3.
